# 00 — Run All (integration entry point)

**Owner:** Marston Ward (AIE / Team Lead) · **Course:** AAI-510 · Group 8

This is the **single-click integration notebook**. It wires together the API
clients (`04`), the agent loop (`03`), and the evaluation harness (`05`) and runs
a small **end-to-end demo in MOCK_MODE** — **zero API keys, zero live Databricks**.

**Run order of the project:** `04_api_clients` → `03_agent_loop` →
`05_evaluation` → this notebook. All shared logic is imported from
`src/soc_agent/`, so nothing is duplicated.


In [1]:
# --- bootstrap: make src/soc_agent importable + default to MOCK_MODE ---
import os, sys
from pathlib import Path

# Walk up to the repo root (the dir that contains src/soc_agent).
_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "src" / "soc_agent").exists():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate src/soc_agent from " + str(_here))

sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)  # so ./mlruns and relative paths are consistent

# Default to zero-creds mock mode unless the grader set real env vars.
os.environ.setdefault("SOC_MOCK_MODE", "1")
os.environ.setdefault("MLFLOW_TRACKING_URI", "file:./mlruns")

from soc_agent import config
SETTINGS = config.get_settings()
print("Repo root        :", REPO_ROOT)
print("MOCK_MODE        :", SETTINGS.mock_mode)
print("LLM provider     :", SETTINGS.llm_provider, "(effective:", SETTINGS.effective_llm_provider() + ")")
print("Live Databricks  :", SETTINGS.use_live_databricks())


Repo root        : /Users/marston.ward/Documents/GitHub/msaai-510-group8-soc-triage-agent
MOCK_MODE        : True
LLM provider     : databricks (effective: mock)
Live Databricks  : False


In [2]:
import json
from soc_agent import config, api_clients, agent, gold_tools, eval_helpers, llm as llm_module

print("Active configuration (secrets redacted):")
for k, v in config.get_settings().summary().items():
    print(f"  {k:28}: {v}")


/Users/marston.ward/Documents/GitHub/msaai-510-group8-soc-triage-agent/_working/.pixi/envs/default/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Active configuration (secrets redacted):
  mock_mode                   : True
  llm_provider (configured)   : databricks
  llm_provider (effective)    : mock
  llm_model                   : databricks-meta-llama-3-1-70b-instruct
  llm_model_b                 : databricks-dbrx-instruct
  llm_base_url                : <unset>
  llm_api_key                 : <unset>
  databricks_host             : <unset>
  databricks_token            : <unset>
  databricks_cluster_id       : <unset>
  databricks_warehouse_id     : <unset>
  use_live_databricks         : False
  vt_api_key                  : <unset>
  shodan_api_key              : <unset>
  nvd_api_key                 : <unset>
  uc_catalog                  : soc_intelligence


## 1) Enrichment clients smoke test

In [3]:
clients = api_clients.build_clients(SETTINGS)
print("VT WORKSTATION6 :", clients["virustotal"].check_ip_reputation("WORKSTATION6")["verdict"])
print("Shodan FILESRV1 :", clients["shodan"].lookup_exposed_ports("FILESRV1")["ports"])
print("NVD RDP         :", [c["cve_id"] for c in clients["nvd"].get_cve_context("RDP", limit=2)])


VT WORKSTATION6 : malicious
Shodan FILESRV1 : [445, 3389, 139]


NVD RDP         : ['CVE-2019-0708']


## 2) End-to-end agent triage (in-scope + out-of-scope)

In [4]:
from soc_agent.mocks import SAMPLE_EVENTS
llm = llm_module.get_llm(settings=SETTINGS)
gold_tools.reset_incident_store()

run = agent.run_triage("Triage the service-creation alert on FILESRV1.", SAMPLE_EVENTS[2],
                       llm=llm, settings=SETTINGS)
print("IN-SCOPE  ->", run["decision"], "|", run["classification"]["tactic"],
      run["classification"]["technique_id"], "| severity", run["classification"]["severity"])

rej = agent.run_triage("Recommend a good sci-fi movie for tonight.", None, llm=llm, settings=SETTINGS)
print("OUT-SCOPE ->", rej["decision"], "::", rej["final_message"][:70], "...")
print("Incidents written:", len(gold_tools.get_incident_store()))


IN-SCOPE  -> escalated | Persistence T1543 | severity HIGH
OUT-SCOPE -> rejected :: This request is outside the SOC triage agent's scope. I only handle se ...
Incidents written: 1


## 3) Evaluation: 5 traces + dual-LLM comparison

In [5]:
gold_tools.reset_incident_store()
traces = eval_helpers.run_traces(settings=SETTINGS)
print("=== 5 TRACES ===")
print(traces[["scenario", "expect", "decision", "tactic", "confidence", "iterations"]].to_string(index=False))

dual = eval_helpers.dual_llm_table(settings=SETTINGS)
print("\n=== DUAL-LLM (same trace, two models) ===")
print(dual[["scenario", "slot", "model", "tactic", "confidence", "priority", "latency_ms"]].to_string(index=False))

print("\n=== SUMMARY ===")
print(eval_helpers.comparison_summary(dual).to_string(index=False))


/Users/marston.ward/Documents/GitHub/msaai-510-group8-soc-triage-agent/_working/.pixi/envs/default/lib/python3.14/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


=== 5 TRACES ===
                    scenario   expect  decision            tactic  confidence  iterations
       credential_access_ws5 escalate escalated Credential Access        0.85           4
    execution_powershell_ws6 escalate escalated         Execution        0.82           4
persistence_service_filesrv1 escalate escalated       Persistence        0.88           4
        out_of_scope_weather   reject  rejected              None         NaN           0
         out_of_scope_recipe   reject  rejected              None         NaN           0



=== DUAL-LLM (same trace, two models) ===
                    scenario    slot                                  model            tactic  confidence  priority  latency_ms
       credential_access_ws5 model_a databricks-meta-llama-3-1-70b-instruct Credential Access        0.85         3       983.2
       credential_access_ws5 model_b               databricks-dbrx-instruct Credential Access        0.78         3       983.8
    execution_powershell_ws6 model_a databricks-meta-llama-3-1-70b-instruct         Execution        0.82         5       990.3
    execution_powershell_ws6 model_b               databricks-dbrx-instruct         Execution        0.75         5      1052.2
persistence_service_filesrv1 model_a databricks-meta-llama-3-1-70b-instruct       Persistence        0.88         4       973.8
persistence_service_filesrv1 model_b               databricks-dbrx-instruct       Persistence        0.81         4       977.2

=== SUMMARY ===
                                 model  mean

## 4) Going live (no code changes — env only)

Default is mock. To run against live Databricks + real model endpoints, set env
vars and restart the kernel (see `docs/SETUP.md`):

```bash
export SOC_MOCK_MODE=0
export LLM_PROVIDER=databricks                       # or openai
export LLM_BASE_URL="https://<workspace>.cloud.databricks.com/serving-endpoints"
export DATABRICKS_HOST="https://<workspace>.cloud.databricks.com"
export DATABRICKS_TOKEN="dapi..."
export DATABRICKS_WAREHOUSE_ID="<sql-warehouse-id>"  # or DATABRICKS_CLUSTER_ID
export LLM_MODEL="databricks-meta-llama-3-1-70b-instruct"
export LLM_MODEL_B="databricks-dbrx-instruct"
# optional: export VT_API_KEY=... SHODAN_API_KEY=...
```

**Kernel:** these notebooks run on the registered Jupyter kernel
**`SOC Agent (Databricks)`** (`soc-agent`).

In [ ]:
print("Integration demo complete — all stages ran in MOCK_MODE with zero creds.")
print("Kernel: SOC Agent (Databricks) [soc-agent]")

Integration demo complete — all stages ran in MOCK_MODE with zero creds.
Kernel: SOC Agent (Databricks) [soc-agent]
